# C-MAPSS RUL pipeline on Microsoft Fabric

This notebook runs the repository's feature pipeline and the LightGBM model as a **Fabric Lakehouse workflow**:

1. **Ingest** the raw C-MAPSS text files from `Files/cmapss/` into Delta tables (bronze)
2. **Feature engineering** with PySpark: regime normalisation and rolling statistics (silver)
3. **Training** with LightGBM, tracked and registered with Fabric's built-in MLflow
4. **Batch scoring** of the test fleet with `MLFlowTransformer`, written to a Delta table that Power BI can read in Direct Lake mode (gold)

**Prerequisites:** a Fabric workspace with a Lakehouse attached to this notebook (Spark runtime 1.3, Python 3.11). Upload the C-MAPSS `train_FD00x.txt`, `test_FD00x.txt` and `RUL_FD00x.txt` files to `Files/cmapss/`.

> This notebook targets the Fabric runtime and is not executed in CI. The same feature code is tested locally against the pandas implementation (`tests/test_spark_parity.py`).

In [ ]:
%pip install -q git+https://github.com/Isaac19911369/predictive-maintenance-rul.git

In [ ]:
SUBSET = "FD004"          # hardest subset: 6 operating regimes, 2 fault modes
WINDOW = 30
RUL_CAP = 125
EXPERIMENT = "cmapss-rul"
MODEL_NAME = "cmapss-rul-lightgbm"

## 1. Bronze: raw files to Delta

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StructField, StructType

from cmapss_rul.data import SENSORS, SETTINGS

schema = StructType(
    [StructField("unit", IntegerType()), StructField("cycle", IntegerType())]
    + [StructField(c, DoubleType()) for c in SETTINGS + SENSORS]
)

def read_raw(kind: str):
    # C-MAPSS files are space-separated with trailing blanks
    lines = spark.read.text(f"Files/cmapss/{kind}_{SUBSET}.txt")
    parts = F.split(F.trim(F.col("value")), r"\s+")
    return lines.select(
        *[parts.getItem(i).cast(f.dataType).alias(f.name) for i, f in enumerate(schema.fields)]
    )

for kind in ("train", "test"):
    read_raw(kind).withColumn("subset", F.lit(SUBSET)) \
        .write.mode("overwrite").format("delta").saveAsTable(f"bronze_cmapss_{kind}")

spark.table("bronze_cmapss_train").groupBy("unit").count().describe("count").show()

## 2. Silver: features with PySpark

In [ ]:
from pyspark.sql import Window

from cmapss_rul import spark_features as sf

train = spark.table("bronze_cmapss_train")
test = spark.table("bronze_cmapss_test")

# piecewise-linear RUL target on run-to-failure trajectories
last = F.max("cycle").over(Window.partitionBy("unit"))
train = train.withColumn("rul", F.least(last - F.col("cycle"), F.lit(RUL_CAP)).cast("double"))

stats = sf.fit_regime_stats(train)                     # fitted on training data only
stats.write.mode("overwrite").format("delta").saveAsTable("silver_cmapss_regime_stats")

# keep sensors that still vary after regime normalisation
train_n = sf.normalize(train, stats)
spread = train_n.select([F.stddev_pop(s).alias(s) for s in SENSORS]).first().asDict()
sensors = [s for s, v in spread.items() if v is not None and v > 1e-4]
print("sensors used:", sensors)

train_f = sf.temporal_features(train_n, sensors, WINDOW)
test_f = sf.temporal_features(sf.normalize(test, stats), sensors, WINDOW)
train_f.write.mode("overwrite").format("delta").saveAsTable("silver_cmapss_train_features")
test_f.write.mode("overwrite").format("delta").saveAsTable("silver_cmapss_test_features")

## 3. Train and register with MLflow

In [ ]:
import lightgbm as lgb
import mlflow
import numpy as np
from mlflow.models.signature import infer_signature

from cmapss_rul.metrics import summary

mlflow.set_experiment(EXPERIMENT)
pdf = spark.table("silver_cmapss_train_features").toPandas()
features = [c for c in pdf.columns if c not in ("unit", "rul")]

# hold out 20% of engines (not rows) for early stopping
rng = np.random.default_rng(42)
units = pdf["unit"].unique()
valid_units = rng.choice(units, size=len(units) // 5, replace=False)
is_valid = pdf["unit"].isin(valid_units)

with mlflow.start_run(run_name=f"lightgbm-{SUBSET}") as run:
    model = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.03, num_leaves=31,
                              min_child_samples=50, subsample=0.8, subsample_freq=1,
                              colsample_bytree=0.8, random_state=42, verbose=-1)
    model.fit(pdf.loc[~is_valid, features], pdf.loc[~is_valid, "rul"],
              eval_set=[(pdf.loc[is_valid, features], pdf.loc[is_valid, "rul"])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
    mlflow.log_params({"subset": SUBSET, "window": WINDOW, "rul_cap": RUL_CAP,
                       "best_iteration": model.best_iteration_, "n_features": len(features)})
    sig = infer_signature(pdf[features].head(100), model.predict(pdf[features].head(100)))
    mlflow.lightgbm.log_model(model, "model", signature=sig, registered_model_name=MODEL_NAME)

    # evaluate on the official test units (last observed cycle of each engine)
    last_rows = (spark.table("silver_cmapss_test_features")
                 .withColumn("is_last", F.col("cycle") == F.max("cycle").over(Window.partitionBy("unit")))
                 .filter("is_last").drop("is_last").orderBy("unit").toPandas())
    y_true = np.minimum(np.loadtxt(f"/lakehouse/default/Files/cmapss/RUL_{SUBSET}.txt"), RUL_CAP)
    pred = np.clip(model.predict(last_rows[features]), 0, RUL_CAP)
    metrics = summary(y_true, pred)
    mlflow.log_metrics(metrics)
    print(metrics)

## 4. Gold: batch scoring for Power BI

In [ ]:
from synapse.ml.predict import MLFlowTransformer

latest = mlflow.MlflowClient().get_latest_versions(MODEL_NAME)[0].version
scorer = MLFlowTransformer(inputCols=features, outputCol="predicted_rul",
                           modelName=MODEL_NAME, modelVersion=latest)

scored = (scorer.transform(spark.table("silver_cmapss_test_features"))
          .withColumn("predicted_rul", F.greatest(F.lit(0.0), F.least(F.lit(float(RUL_CAP)), "predicted_rul")))
          .withColumn("maintenance_window",
                      F.when(F.col("predicted_rul") < 30, "urgent")
                       .when(F.col("predicted_rul") < 60, "plan")
                       .otherwise("healthy"))
          .withColumn("scored_at", F.current_timestamp())
          .withColumn("model_version", F.lit(latest)))

scored.write.mode("overwrite").format("delta").saveAsTable("gold_cmapss_rul_predictions")
display(scored.filter("maintenance_window = 'urgent'").groupBy("unit").agg(F.min("predicted_rul")))